**Imports**

# Obligatorio - Taller Agentes Inteligentes 2025

## Ramiro Sanes () - Joaquin Guerra ()

En este trabajo obligatorio aplicaremos los conceptos vistos en el curso para diseñar, implementar y evaluar agentes capaces de aprender a jugar al clásico **Breakout** de Atari, utilizando el entorno provisto por Farama Gymnasium ([https://ale.farama.org/environments/breakout/](https://ale.farama.org/environments/breakout/)). 

<p align="center">
  <img src="https://media.tenor.com/oMxHgRrISJsAAAAM/atari-deep-learning.gif" alt="Atari Deep Learning"/>
</p>


El ejercicio se enmarca en un contexto de aprendizaje práctico, donde trabajaremos con las interfaces estándar de Gymnasium para:

1. **Profundizar en algoritmos de valor**: implementaremos y compararemos dos variantes de Q-Learning basadas en redes neuronales profundas:
   * **Deep Q Learning (DQN)**
   * **Double Deep Q Learning (DDQN)**
2. **Evaluar rendimiento y estabilidad**: registraremos las recompensas obtenidas durante el entrenamiento de cada agente y analizaremos su comportamiento mediante gráficas comparativas.
3. **Demostrar resultados de forma visual**: capturaremos vídeos que muestren a cada agente “resolviendo” el entorno, entendido como la habilidad de romper al menos cinco bloques en una partida.

Debido a las limitaciones de tiempo y cómputo propias de un entorno de curso, no se espera entrenar modelos durante más de diez horas por agente. Por ello, será fundamental:

* Integrar puntos de **checkpoint** para guardar periódicamente los pesos de la red.
* Seguir en los puntos 2 y 3 la arquitectura y técnicas originales propuestas en los papers seminales de DQN y DDQN, dejando la experimentación adicional para el punto extra.
* Flexibilizar la notebook de guía: pueden reorganizarla o dividirla en múltiples archivos según su conveniencia.


## Setup

Importamos las librerías, clases y funciones necesarias para llevar a cabo las tareas.

__En el archivo utils.py__:

- Incorporamos la función __process_state__ que transforma las observaciones en tensores y le sumamos como parámetro el device.
- A la función __make_env__ se le agregó un parámetro de tipo booleano que recibe "True" por defecto, y en ese caso utiliza el Wrapper de gymnasium __FireOnLifeLostWrapper__ que automáticamente ejecuta la accion Fire luego de cada perdida de vida y cada reset ya que en una primera instancia nos encontramos con que el agente no aprendía ya que no "disparaba" por lo que este paso nos ayudó a mejorar en el aprendizaje.


__En el archivo dqn_cnn_model__:

- Importamos el modelo para aproximar la función de valor $Q(s,a)$, definido como una red neuronal, Inspirado en Mnih et al. (2013) ([arXiv:1312.5602](https://arxiv.org/abs/1312.5602)). El modelo consiste en una red convolucional que recibe como estados 4 frames consecutivos del entorno Atari reescaladas a 84x84.
    - La primera capa oculta de la red se compone de 16 kernels de 8x8 con un stride de 4 seguido de una activacion ReLU.
    - La segunda contiene 32 kernels de 4x4 con un stride de 2 tambien seguido de una activación ReLU.
    - La última capa oculta es una fully-connected que contiene 256 unidades provenientes de aplanar la salida de la capa anterior, con activación ReLu
    - La capa de salida recibe 256 features de entrada de la capa anterior, y tiene cómo salida la cantidad de acciones, que son 4 en nuestro caso.

## Los agentes

Se definieron 2 agentes:

- El __DQNAgent__, implementa Deep Q-Learning clásico, buscando aprender la funcion de valor $Q(s,a)$ .
Este agente utiliza una sola red neuronal definida en la celda anterior. Recibe el frame correspondiente a un estado y devuelve una estimación del valor para cada una de las 4 acciones posibles en ese estado.
Tanto este agente como el siguiente utilizan una __Replay Memory__ que guarda las transiciones (estado, accion, recompensa, finalizado, estado_siguiente)
Para actualizar los pesos de la red no se le provee la experiencia inmediata de la interacción con el ambiente, si no que se muestrean batches de 32 transiciones de la memoria buscando romper con la correlacion entre experiencias consecutivas.

- El __DoubleDQNAgent__ busca mejorar al anterior agente solucionando el sesgo de máximización de los valores de Q.
Para esto utiliza otra red (target_net) que se actualiza cada "sync_target" pasos, definido como 1000 por defecto, lo que hace que el target con el que actualizamos la funcion $Q(s,a)$ sea más estable.

## Memoria

Para la implementación de la replay memory utilizamos una memoria circular que recibe su capacidad como parámetro en su creación ("capacity")
La memoria almacena las transiciones. Las agrega en caso que la capacidad actual sea menor a la capacity definida y las reemplaza por la más antigua en el caso de que la capacidad actual sea igual a capacity.
El método add es el que se encarga de agregar las transiciones a memoria, mientras que el método sample se encarga de seleccionar un bache segun el tamaño ("batch_size") recido como parámetro.

El archivo replay memory cuenta con las funciones __add_with_priority__ , __sample_with_priority__ y __update_priorities__ que posteriormente serán utilizadas al implementar la replay memory con prioridad (PER) para experimentar.

In [1]:
import os
import torch
import numpy as np
import random
import numpy as np
import gymnasium
import ale_py
from IPython.display import Video

from utils import make_env, process_state, show_observation_stack, plot_rewards_and_max_values
from dqn_agent import DQNAgent
from dqn_cnn_model import DQN_CNN_Model
from double_dqn_agent import DoubleDQNAgent

In [2]:
SEED = 23

torch.manual_seed(SEED)
torch.backends.cudnn.deterministic=True # https://discuss.pytorch.org/t/what-is-the-differenc-between-cudnn-deterministic-and-cudnn-benchmark/38054
torch.backends.cudnn.benchmark=True # https://discuss.pytorch.org/t/what-does-torch-backends-cudnn-benchmark-do/5936/4
np.random.seed(SEED)
random.seed(SEED)

In [3]:
DEVICE = "cpu"
if torch.cuda.is_available():
    DEVICE = "cuda"  
elif torch.backends.mps.is_available():
    DEVICE = "mps" 

In [4]:
GRAY_SCALE = True 
SCREEN_SIZE = 84 
NUM_STACKED_FRAMES = 4 
SKIP_FRAMES = 4 
ENV_NAME = "ALE/Breakout-v5" 

## Entrenamiento

### Entrenamos al agente DQNAgent y medimos sus resultados

In [5]:
#Hiperparámetros de entrenamiento del agente DQN
TOTAL_STEPS = 10_000_000
#EPISODES = 20_000
EPISODES = 10
STEPS_PER_EPISODE = 20_000

EPSILON_INI = 1
EPSILON_MIN = 0.05
EPSILON_ANNEAL_STEPS = 1_000_000

EPISODE_BLOCK = 100

BATCH_SIZE = 32
BUFFER_SIZE = 50_000

GAMMA = 0.995
LEARNING_RATE = 1e-5

Instanciamos el environment, grabando los videos durante el entrenamiento cada 500 episodios.
El agente entrenará durante 20.000 episdios con un máximo de 10.000.000 de pasos totales.

Epsilon irá disminuyendo desde 1 (política totalmente exploratoria) hasta 0.05 durante 1.000.000 de pasos

Se utiliza un gamma de 0.995 como ponderador de recompensas fururas y un learning rate de 1e-5 como tasa de aprendizaje del optimizador.

El batch size y buffer size se utilizan para definir la capacidad de la replay memory (50.000 en este caso) y determinar el tamaño de los batches a samplear (32)

In [6]:
env = make_env(ENV_NAME,
                video_folder='./videos/dqn_training',
                name_prefix="breakout",
                record_every=500,
                grayscale=GRAY_SCALE,
                screen_size=SCREEN_SIZE,
                stack_frames=NUM_STACKED_FRAMES,
                skip_frames=SKIP_FRAMES
                )

net = DQN_CNN_Model(env.observation_space.shape, env.action_space.n).to(DEVICE)

dqn_agent = DQNAgent(env, net, process_state, BUFFER_SIZE, BATCH_SIZE, LEARNING_RATE, GAMMA, 
                     epsilon_i=EPSILON_INI, epsilon_f=EPSILON_MIN, 
                     epsilon_anneal_steps=EPSILON_ANNEAL_STEPS, 
                     episode_block=EPISODE_BLOCK, device=DEVICE)



c:\Users\joaco\anaconda3\envs\obl_taller_ia\Lib\site-packages\gymnasium\wrappers\rendering.py:283: UserWarning: WARN: Overwriting existing videos at c:\Users\joaco\Documents\ort\entregas_taller_de_ia\obligatorio\videos\dqn_training folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Generamos una lista con 10 estados procesados al azar, para evaluar el valor de las funciones en esos estados durante el entrenamiento. Utilizamos los mismos 10 estados para evaluar a todos los agentes.

In [7]:
state_eval_list = []
for _ in range(10):
    state_eval_list.append(process_state(env.reset()[0],DEVICE))


Los videos del entrenamiento serán guardados en './videos/dqn_training' y se guardarán los pesos de la red cada 1000 pasos.

In [ ]:
rewards_1, state_values_1 = dqn_agent.train(EPISODES, STEPS_PER_EPISODE, TOTAL_STEPS, random_states=state_eval_list)

Training:   0%|          | 0/10 [00:00<?, ?episode/s]c:\Users\joaco\Documents\ort\entregas_taller_de_ia\obligatorio\dqn_agent.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(s, dtype=torch.float32).to(self.device)
c:\Users\joaco\Documents\ort\entregas_taller_de_ia\obligatorio\dqn_agent.py:123: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(s, dtype=torch.float32).to(self.device)
Training: 100%|██████████| 10/10 [00:30<00:00,  3.07s/episode, reward=0.9, epsilon=0.998, steps=1586] 


In [10]:
plot_rewards_and_max_values(rewards_1, state_values_1, "DQN Agent", "Evaluación del entrenamiento",)

NameError: name 'rewards_1' is not defined

## Resultados

Hacemos que el agente juegue 10 episodios y mostramos el desempeño

In [7]:
env = make_env(ENV_NAME,
                video_folder='./videos/dqn_test',
                name_prefix="breakout",
                record_every=1,
                grayscale=GRAY_SCALE,
                screen_size=SCREEN_SIZE,
                stack_frames=NUM_STACKED_FRAMES,
                skip_frames=SKIP_FRAMES
                )

c:\Users\joaco\anaconda3\envs\obl_taller_ia\Lib\site-packages\gymnasium\wrappers\rendering.py:283: UserWarning: WARN: Overwriting existing videos at c:\Users\joaco\Documents\ort\entregas_taller_de_ia\obligatorio\videos\dqn_test folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


In [ ]:
checkpoint_path = "checkpoint_20000"
print(f"Cargando checkpoint desde {checkpoint_path}")
dqn_agent.load_checkpoint(checkpoint_path)

retornos = dqn_agent.play(env=env, episodes=10)


video_path = f"./videos/dqn_test/breakout-episode-{retornos.argmax()}.mp4"

print(f"El agente DQNAgent obtuvo un retorno promedio de {retornos.mean()} en {len(retornos)} episodios. \n Y el mejor retorno fue {retornos.max()} en el episodio {retornos.argmax()}.")

# Muestra el vídeo
Video(video_path, embed=True, width=600)

Cargando checkpoint desde checkpoint_10000_policy_net.pth
Checkpoint loaded from weights/dqn/checkpoint_10000_policy_net.pth
El agente DQNAgent obtuvo un retorno promedio de 19.2 en 10 episodios. 
 Y el mejor retorno fue 27.0 en el episodio 1.


### Entrenamos al agente DoubleDQNAgent y medimos sus resultados

Utilizamos los mismos hiperparámetros que el agente DQNAgent.
En este caso, por lo explicado en la definición del agente, debemos instanciar una nueva red "target_net" y se la pasamos como parámetro al agente.

In [ ]:
env = make_env(ENV_NAME,
                video_folder='./videos/ddqn_training',
                name_prefix="breakout",
                record_every=500,
                grayscale=GRAY_SCALE,
                screen_size=SCREEN_SIZE,
                stack_frames=NUM_STACKED_FRAMES,
                skip_frames=SKIP_FRAMES
                )

net = DQN_CNN_Model(env.observation_space.shape, env.action_space.n).to(DEVICE)
target_net = DQN_CNN_Model(env.observation_space.shape, env.action_space.n).to(DEVICE)

double_dqn_agent = DoubleDQNAgent(env, net,target_net, process_state, BUFFER_SIZE, BATCH_SIZE, LEARNING_RATE, GAMMA, 
                     epsilon_i=EPSILON_INI, epsilon_f=EPSILON_MIN, 
                     epsilon_anneal_steps=EPSILON_ANNEAL_STEPS, 
                     episode_block=EPISODE_BLOCK, device=DEVICE)



c:\Users\joaco\anaconda3\envs\obl_taller_ia\Lib\site-packages\gymnasium\wrappers\rendering.py:283: UserWarning: WARN: Overwriting existing videos at c:\Users\joaco\Documents\ort\entregas_taller_de_ia\obligatorio\videos\dqn_training folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
c:\Users\joaco\anaconda3\envs\obl_taller_ia\Lib\site-packages\gymnasium\wrappers\rendering.py:416: UserWarning: WARN: Unable to save last video! Did you call close()?
  logger.warn("Unable to save last video! Did you call close()?")


In [ ]:
rewards_2, state_values_2 = double_dqn_agent.train(EPISODES, STEPS_PER_EPISODE, TOTAL_STEPS,random_states=state_eval_list)

Training:   0%|          | 9/20000 [00:34<21:06:22,  3.80s/episode, reward=1.11, epsilon=0.999, steps=1501] 


KeyboardInterrupt: 

In [ ]:
plot_rewards_and_max_values(rewards_1, state_values_1, "DQN Agent", "Evaluación del entrenamiento",)

Hacemos que el agente juegue 10 episodios y mostramos el desempeño

In [ ]:

env = make_env(ENV_NAME,
                video_folder='./videos/ddqn_test',
                name_prefix="breakout",
                record_every=1,
                grayscale=GRAY_SCALE,
                screen_size=SCREEN_SIZE,
                stack_frames=NUM_STACKED_FRAMES,
                skip_frames=SKIP_FRAMES
                )

checkpoint_path = "checkpoint_20000"
print(f"Cargando checkpoint desde {checkpoint_path}")
double_dqn_agent.load_checkpoint(checkpoint_path)

retornos = double_dqn_agent.play(env=env, episodes=10)


video_path = f"./videos/ddqn_test/breakout-episode-{retornos.argmax()}.mp4"

print(f"El agente DoubleDQNAgent obtuvo un retorno promedio de {retornos.mean()} en {len(retornos)} episodios. \n Y el mejor retorno fue {retornos.max()} en el episodio {retornos.argmax()}.")

# Muestra el vídeo
Video(video_path, embed=True, width=600)

c:\Users\joaco\anaconda3\envs\obl_taller_ia\Lib\site-packages\gymnasium\wrappers\rendering.py:283: UserWarning: WARN: Overwriting existing videos at c:\Users\joaco\Documents\ort\entregas_taller_de_ia\obligatorio\videos\ddqn_test folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Cargando checkpoint desde checkpoint_2000
Checkpoints loaded from weights/ddqn/checkpoint_2000_policy_net.pth and weights/ddqn/checkpoint_2000_target_net.pth
El agente DQNAgent obtuvo un retorno promedio de 5.8 en 10 episodios. 
 Y el mejor retorno fue 11.0 en el episodio 2.
